# Socratic Debate Framework on GSM8K

**Models:**
- **Proponent**: `llama3.1:8b` (Ollama, local)
- **Prober**: `qwen2.5:8b` (Ollama, local)
- **Judge**: `openai/gpt-oss-20b` (Groq API)

Make sure you select a **GPU runtime** (T4 is fine) in Colab: Runtime > Change runtime type > GPU

## 1. Install Ollama & Pull Models

In [1]:
!sudo apt-get update -qq && sudo apt-get install -y -qq zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:114

In [2]:
import subprocess, time
proc = subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)
print(f'Ollama server started (pid={proc.pid})')

Ollama server started (pid=3642)


In [3]:
!ollama pull llama3.1:8b
!ollama pull qwen2.5:7b

In [4]:
!ollama list

NAME           ID              SIZE      MODIFIED           
qwen2.5:7b     845dbda0ea48    4.7 GB    12 seconds ago        
llama3.1:8b    46e0c10c039e    4.9 GB    About a minute ago    


## 2. Install Python Dependencies

In [5]:
!pip install -q groq datasets requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 7.6 MB/s eta 0:00:00


## 3. Set API Keys

In [ ]:
import os
os.environ['GROQ_API_KEY'] = ''  # <-- Paste your Groq API key here

## 4. Create Project Files

In [7]:
import os
os.makedirs('NLP/core', exist_ok=True)
os.makedirs('NLP/outputs', exist_ok=True)

In [8]:
%%writefile NLP/core/ollama_llm.py
import requests

OLLAMA_URL = "http://localhost:11434/api/chat"


def call_ollama_model(
    model: str,
    system_prompt: str,
    user_prompt: str,
    temperature: float = 0.0,
    max_tokens: int = 1024,
) -> str:
    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        "stream": False,
        "options": {
            "temperature": temperature,
            "num_predict": max_tokens,
        },
    }

    resp = requests.post(OLLAMA_URL, json=payload, timeout=300)
    if resp.status_code != 200:
        raise RuntimeError(f"Ollama error {resp.status_code}: {resp.text}")

    data = resp.json()

    if isinstance(data.get("message"), dict):
        content = data["message"].get("content")
        if isinstance(content, str):
            return content.strip()

    if isinstance(data.get("response"), str):
        return data["response"].strip()

    raise RuntimeError(f"Unexpected response format: {data}")

Writing NLP/core/ollama_llm.py


In [11]:
%%writefile /content/NLP/core/groq_llm.py
import os
import time
from groq import Groq

_client = Groq(api_key=os.environ["GROQ_API_KEY"])

MAX_RETRIES = 6
BASE_WAIT = 10


def call_groq_model(
    model: str,
    system_prompt: str,
    user_prompt: str,
    temperature: float = 0.0,
    top_p: float = 1.0,
    max_tokens: int | None = None,
) -> str:
    kwargs: dict = dict(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=temperature,
        top_p=top_p,
    )
    if max_tokens is not None:
        kwargs["max_tokens"] = max_tokens

    for attempt in range(MAX_RETRIES):
        try:
            resp = _client.chat.completions.create(**kwargs)
            return resp.choices[0].message.content or ""
        except Exception as e:
            if "429" in str(e) or "rate" in str(e).lower():
                wait = BASE_WAIT * (2 ** attempt)
                print(f"  Rate limited, waiting {wait}s (attempt {attempt + 1}/{MAX_RETRIES})...")
                time.sleep(wait)
            else:
                raise

    raise RuntimeError(f"Rate limit exceeded after {MAX_RETRIES} retries")

Writing /content/NLP/core/groq_llm.py


In [13]:
%%writefile /content/NLP/proponent.py
import json
import os
import re
from typing import Any

from core.ollama_llm import call_ollama_model

PROPONENT_MODEL = os.environ.get("PROPONENT_MODEL", "llama3.1:8b")

INITIAL_SYSTEM_PROMPT = """
You are a careful math solver.

Rules:
- Extract the given quantities first.
- Use only the information in the question.
- Do not write long explanations.
- Compute in at most 6 short steps.
- If someone drives away from home and later heads back toward home, track direction: motion toward home
  shrinks distance from home; motion away grows it. Do not add all segment lengths as if they were
  all in the same direction.
- For questions that ask how far someone is from home (or from a fixed place) at the end, compute net
  distance from that place after each leg -- not the sum of every trip segment unless every segment
  moves farther from that place.
- After turning toward home: if D is miles from home before a leg and the leg covers L miles toward
  home, then afterward distance from home is D - L (subtraction), never D + L. Standstill adds 0
  toward home, so D is unchanged.
- For percent increases or decreases ("by 150%", "increased by 20%", etc.), name what the percentage
  is **of** (the base). The base is whatever the wording ties the change to (e.g. "value of the
  house" -> the house's stated value), not automatically purchase price plus every later expense unless
  the problem explicitly says the percent applies to that combined total.
- Profit or net gain is usually final value minus **all** money spent; that total cost can differ from
  the **percentage base** used for a "value increased by P%" phrase -- keep both straight.
- Identify what the question asks for: **total** for the whole group vs **per** person/animal/unit, time
  vs distance, etc. Your **Final answer** must match that target (e.g. total cups for the meal, not cups
  per chicken, unless the question asks per-chicken).
- If the problem states a uniform rate for each member (e.g. each chicken 3 cups/day), then
  (number of members x that rate) is the correct total daily amount unless the text says otherwise.
  Do not abandon that for a wrong quantity type when responding to a challenge.
- Return ONLY a JSON object with exactly these keys:
  - "steps": an array of short strings
  - "final_answer": a number

Do NOT include any text outside the JSON.
""".strip()

REVISION_SYSTEM_PROMPT = """
You are a careful math solver responding to a Socratic challenge.

The prober has asked a "Why?" question about one of your reasoning steps.
You MUST do two things:
1. Directly answer the prober's question in the "response_to_probe" field -- explain the
   reasoning or justification for the challenged step in 1-3 sentences.
2. If the challenge reveals a genuine error, fix your steps. Otherwise keep the math the same
   but do NOT copy your previous steps word-for-word -- rephrase or reorganize to show you
   considered the challenge.

Rules:
- Use only the information in the original question.
- Compute in at most 6 short steps.
- If someone drives away from home and later heads back toward home, track direction: motion toward home
  shrinks distance from home; motion away grows it.
- For percent increases or decreases, name what the percentage is **of** (the base).
- Profit or net gain is usually final value minus **all** money spent.
- Identify what the question asks for: total vs per-unit, time vs distance, etc.
- Return ONLY a JSON object with exactly these keys:
  - "response_to_probe": a string directly answering the prober's question
  - "steps": an array of short strings (rephrased, not copied verbatim)
  - "final_answer": a number

Do NOT include any text outside the JSON.
""".strip()


def _fix_trailing_commas(text):
    return re.sub(r",\s*([}\]])", r"\1", text)


def _strip_code_fences(text):
    text = re.sub(r"^```(?:json)?\s*\n?", "", text, flags=re.MULTILINE)
    text = re.sub(r"\n?```\s*$", "", text, flags=re.MULTILINE)
    return text.strip()


def _extract_json(text):
    text = _strip_code_fences(text.strip())
    for candidate in [text, _fix_trailing_commas(text)]:
        try:
            parsed = json.loads(candidate)
            if isinstance(parsed, dict):
                return parsed
        except json.JSONDecodeError:
            pass
    start = text.find("{")
    end = text.rfind("}")
    if start == -1 or end == -1 or end <= start:
        raise ValueError(f"Model did not return valid JSON: {text}")
    snippet = _fix_trailing_commas(text[start : end + 1])
    parsed = json.loads(snippet)
    if not isinstance(parsed, dict):
        raise ValueError("Expected JSON object from model")
    return parsed


def ask_proponent(question, critique=None, previous_answer=None, transcript=None):
    is_revision = critique is not None
    system_prompt = REVISION_SYSTEM_PROMPT if is_revision else INITIAL_SYSTEM_PROMPT
    parts = [f"Original question:\n{question}"]
    if previous_answer is not None:
        parts.append("Your previous answer:\n" + json.dumps(previous_answer, ensure_ascii=False, indent=2))
    if critique:
        parts.append(
            f'The prober challenges:\n"{critique}"\n\n'
            'You MUST answer this specific question in "response_to_probe".\n'
            'Then provide your (rephrased) steps and final_answer.'
        )
    if transcript:
        parts.append("Transcript so far:\n" + "\n".join(f"{item['role'].upper()}: {item['text']}" for item in transcript))
    if is_revision:
        parts.append('Return JSON with keys "response_to_probe", "steps", and "final_answer".')
    else:
        parts.append('Return JSON with keys "steps" and "final_answer".')
    user_prompt = "\n\n".join(parts)
    raw = call_ollama_model(PROPONENT_MODEL, system_prompt, user_prompt, temperature=0.0 if is_revision else 0.3, max_tokens=1024)
    return _extract_json(raw)

Writing /content/NLP/proponent.py


In [14]:
%%writefile /content/NLP/prober.py
import json
import os
import re
from typing import Any

from core.ollama_llm import call_ollama_model

PROBER_MODEL = os.environ.get("PROBER_MODEL", "qwen2.5:7b")


def _strip_think_tags(text):
    return re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()


SYSTEM_PROMPT = """
You are an arithmetic-checking Socratic skeptic.

BEFORE asking your question, you MUST silently verify every computation in the proponent's
steps. Re-do each arithmetic operation yourself. If any step produces a wrong number,
that is the step you must challenge.

Procedure:
1. Read the original problem and extract the given numbers.
2. Walk through each of the proponent's steps. For every arithmetic operation
   (add, subtract, multiply, divide, percentage), compute the correct result yourself.
3. If you find a step whose result is wrong, ask about THAT step.
4. If all arithmetic is correct, pick the least-justified reasoning step.

Output rules:
- Ask exactly ONE question.
- Start with "Why does" or "Why is".
- Do NOT include your verification work, commentary, or explanations.
- Do NOT question facts stated directly in the problem.
- Do NOT repeat or paraphrase a previous probe (check the transcript).
- Use a different reasoning category each turn (arithmetic error, aggregation logic,
  unit/quantity interpretation, question-target mismatch).
- Keep it short. End with exactly one question mark.
- Output ONLY the question.
""".strip()


def ask_prober(question, proponent_output, transcript=None):
    transcript_text = ""
    if transcript:
        transcript_text = "\n".join(f"{item['role'].upper()}: {item['text']}" for item in transcript)
    user_prompt = f"""
Original question:
{question}

Proponent output:
{json.dumps(proponent_output, ensure_ascii=False, indent=2)}

Transcript so far:
{transcript_text if transcript_text else '(none)'}

Ask one concise challenge question that targets a specific step.
""".strip()
    raw = call_ollama_model(model=PROBER_MODEL, system_prompt=SYSTEM_PROMPT, user_prompt=user_prompt, temperature=0.2, max_tokens=256)
    return _strip_think_tags(raw)

Writing /content/NLP/prober.py


In [15]:
%%writefile /content/NLP/debate_loop.py
import json
from collections import Counter
from typing import Any

from proponent import ask_proponent
from prober import ask_prober


def run_debate(question, gold_answer=None, max_turns=3):
    transcript = []
    rounds = []
    current_json = ask_proponent(question)
    initial_answer = str(current_json.get("final_answer", ""))
    current_answer = initial_answer
    transcript.append({"role": "proponent", "text": json.dumps(current_json, ensure_ascii=False)})
    answer_history = [initial_answer]
    for turn in range(1, max_turns + 1):
        probe = ask_prober(question=question, proponent_output=current_json, transcript=transcript).strip()
        transcript.append({"role": "prober", "text": probe})
        revised_json = ask_proponent(question=question, critique=probe, previous_answer=current_json, transcript=transcript)
        current_answer = str(revised_json.get("final_answer", ""))
        answer_history.append(current_answer)
        transcript.append({"role": "proponent", "text": json.dumps(revised_json, ensure_ascii=False)})
        rounds.append({"turn": turn, "probe": probe, "proponent_answer": revised_json})
        current_json = revised_json
    counts = Counter(answer_history)
    majority_answer, majority_count = counts.most_common(1)[0]
    if majority_count > len(answer_history) / 2:
        final_answer = majority_answer
    else:
        final_answer = initial_answer
    return {
        "question": question, "gold_answer": gold_answer,
        "initial_answer": initial_answer, "final_answer": final_answer,
        "raw_final_answer": current_answer, "answer_history": answer_history,
        "safeguard_applied": final_answer != current_answer,
        "final_proponent_text": json.dumps(current_json, ensure_ascii=False),
        "transcript": transcript, "rounds": rounds, "num_turns": max_turns,
    }

Writing /content/NLP/debate_loop.py


In [16]:
%%writefile /content/NLP/run_debate.py
import json
import time
from pathlib import Path

from debate_loop import run_debate


def load_jsonl(path):
    data = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                data.append(json.loads(line))
    return data


def main():
    input_path = Path("gsm8k_sample.jsonl")
    output_path = Path("outputs/debate_results_new.json")
    output_path.parent.mkdir(parents=True, exist_ok=True)
    dataset = load_jsonl(input_path)
    results = []
    for idx, sample in enumerate(dataset):
        question = sample["question"]
        gold_answer = sample.get("answer")
        print(f"Running sample {idx}...")
        try:
            debate = run_debate(question=question, gold_answer=gold_answer, max_turns=3)
            status = "ok"
            error = None
        except Exception as e:
            debate = {}
            status = "error"
            error = str(e)
        record = {
            "id": sample.get("id", idx),
            "question": question,
            "debate": debate if status == "ok" else {"error": error},
            "status": status,
        }
        results.append(record)
        if idx < len(dataset) - 1:
            time.sleep(2)
    with output_path.open("w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    print(f"Saved results to {output_path}")


if __name__ == "__main__":
    main()

Writing /content/NLP/run_debate.py


In [17]:
%%writefile /content/NLP/judge.py
import json
import os
import re
from decimal import Decimal, InvalidOperation
from pathlib import Path
from typing import Any
from groq import Groq

INPUT_PATH = Path("outputs/debate_results_new.json")
OUTPUT_PATH = Path("outputs/judged_results_new.json")
JUDGE_MODEL = os.environ.get("GROQ_JUDGE_MODEL", "openai/gpt-oss-20b")
client = Groq(api_key=os.environ.get("GROQ_API_KEY"))


def normalize_numeric(text):
    s = str(text).strip().replace(",", "")
    m = re.search(r"-?\d+(?:\.\d+)?", s)
    return m.group(0) if m else s


def numeric_equal(a, b):
    na, nb = normalize_numeric(a), normalize_numeric(b)
    try:
        return Decimal(na) == Decimal(nb)
    except (InvalidOperation, ValueError):
        return na == nb


def safe_json_loads(text):
    try:
        obj = json.loads(text)
        if isinstance(obj, dict):
            return obj
    except json.JSONDecodeError:
        pass
    return {}


def coerce_sample(obj):
    if isinstance(obj, dict):
        return obj
    if isinstance(obj, str):
        try:
            parsed = json.loads(obj)
        except json.JSONDecodeError:
            return None
        if isinstance(parsed, dict):
            return parsed
        if isinstance(parsed, str):
            try:
                parsed2 = json.loads(parsed)
                if isinstance(parsed2, dict):
                    return parsed2
            except json.JSONDecodeError:
                return None
    return None


def load_samples(path):
    raw = path.read_text(encoding="utf-8").strip()
    if not raw:
        return []
    try:
        data = json.loads(raw)
        if isinstance(data, list):
            out = []
            for item in data:
                sample = coerce_sample(item)
                if sample is not None:
                    out.append(sample)
            return out
        sample = coerce_sample(data)
        return [sample] if sample is not None else []
    except json.JSONDecodeError:
        pass
    samples = []
    with path.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                item = json.loads(line)
            except json.JSONDecodeError as e:
                print(f"Skipping invalid JSON on line {line_no}: {e}")
                continue
            sample = coerce_sample(item)
            if sample is None:
                print(f"Skipping non-object on line {line_no}")
                continue
            samples.append(sample)
    return samples


def extract_debate(sample):
    debate = sample.get("debate", {})
    return debate if isinstance(debate, dict) else {}


def extract_final_answer(sample):
    debate = extract_debate(sample)
    final_answer = debate.get("final_answer", "")
    if final_answer not in ("", None):
        return final_answer
    fpt = debate.get("final_proponent_text", "")
    if isinstance(fpt, str) and fpt.strip():
        parsed = safe_json_loads(fpt)
        for key in ("Final answer", "final answer", "Final Answer", "answer", "Answer"):
            if key in parsed:
                return parsed[key]
        m = re.findall(r"-?\d+(?:\.\d+)?", fpt.replace(",", ""))
        if m:
            return m[-1]
    return ""


def extract_gold_answer(sample):
    if "gold_answer" in sample and sample["gold_answer"] not in ("", None):
        return sample["gold_answer"]
    debate = extract_debate(sample)
    for key in ("gold_answer", "answer", "correct_answer"):
        val = debate.get(key)
        if val not in ("", None):
            return val
    return ""


JUDGE_SYSTEM_PROMPT = """
You are a strict reasoning judge for math debate transcripts.

Evaluate the model's transcript against the question and gold answer.

You must score:
- answer_correct: whether the final answer matches the gold answer
- reasoning_score: overall reasoning quality from 1 to 5
- probe_response_score: how well the proponent responds to the prober from 1 to 5
- consistency_score: whether the transcript stays consistent from 1 to 5
- overall_judgment: "pass" if the final answer is correct and reasoning_score >= 3, otherwise "fail"
- notes: a short explanation

Use these guidelines:
- 5 = excellent
- 4 = mostly correct
- 3 = mixed
- 2 = weak
- 1 = invalid

Be strict.
""".strip()

JUDGE_SCHEMA = {
    "type": "object",
    "properties": {
        "answer_correct": {"type": "boolean"},
        "reasoning_score": {"type": "integer", "minimum": 1, "maximum": 5},
        "probe_response_score": {"type": "integer", "minimum": 1, "maximum": 5},
        "consistency_score": {"type": "integer", "minimum": 1, "maximum": 5},
        "overall_judgment": {"type": "string", "enum": ["pass", "fail"]},
        "notes": {"type": "string"},
    },
    "required": ["answer_correct", "reasoning_score", "probe_response_score", "consistency_score", "overall_judgment", "notes"],
    "additionalProperties": False,
}


def judge_sample(sample):
    sample = coerce_sample(sample) or {}
    debate = extract_debate(sample)
    final_answer = extract_final_answer(sample)
    gold_answer = extract_gold_answer(sample)
    payload = {
        "question": sample.get("question", ""),
        "gold_answer": gold_answer,
        "final_answer": final_answer,
        "initial_answer": debate.get("initial_answer", ""),
        "final_proponent_text": debate.get("final_proponent_text", ""),
        "transcript": debate.get("transcript", []),
    }
    response = client.chat.completions.create(
        model=JUDGE_MODEL,
        messages=[{"role": "system", "content": JUDGE_SYSTEM_PROMPT}, {"role": "user", "content": json.dumps(payload, ensure_ascii=False)}],
        temperature=0, top_p=1,
        response_format={"type": "json_schema", "json_schema": {"name": "debate_judge", "strict": True, "schema": JUDGE_SCHEMA}},
    )
    raw_text = response.choices[0].message.content or "{}"
    judge_obj = safe_json_loads(raw_text)
    expected_correct = False
    if gold_answer not in ("", None) and final_answer not in ("", None):
        expected_correct = numeric_equal(final_answer, gold_answer)
    if "answer_correct" not in judge_obj:
        judge_obj["answer_correct"] = expected_correct
    if "reasoning_score" not in judge_obj:
        judge_obj["reasoning_score"] = 1 if not judge_obj["answer_correct"] else 3
    if "probe_response_score" not in judge_obj:
        judge_obj["probe_response_score"] = 1 if not judge_obj["answer_correct"] else 3
    if "consistency_score" not in judge_obj:
        judge_obj["consistency_score"] = 1 if not judge_obj["answer_correct"] else 3
    if "overall_judgment" not in judge_obj:
        judge_obj["overall_judgment"] = "pass" if (judge_obj["answer_correct"] and int(judge_obj["reasoning_score"]) >= 2) else "fail"
    if "notes" not in judge_obj:
        judge_obj["notes"] = "Fallback verdict."
    sample["judge"] = judge_obj
    sample["judge_match"] = bool(judge_obj.get("answer_correct")) == expected_correct if gold_answer not in ("", None) and final_answer not in ("", None) else bool(judge_obj.get("answer_correct"))
    sample["judge_answer_correct"] = bool(judge_obj.get("answer_correct"))
    sample["judge_reasoning_score"] = int(judge_obj.get("reasoning_score", 0))
    sample["judge_probe_response_score"] = int(judge_obj.get("probe_response_score", 0))
    sample["judge_consistency_score"] = int(judge_obj.get("consistency_score", 0))
    sample["judge_overall_judgment"] = judge_obj.get("overall_judgment", "fail")
    return sample


def main():
    if not os.environ.get("GROQ_API_KEY"):
        raise RuntimeError("GROQ_API_KEY is not set.")
    if not INPUT_PATH.exists():
        raise FileNotFoundError(f"Input file not found: {INPUT_PATH}")
    OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    samples = load_samples(INPUT_PATH)
    with OUTPUT_PATH.open("w", encoding="utf-8") as f_out:
        for idx, sample in enumerate(samples, start=1):
            judged = judge_sample(sample)
            f_out.write(json.dumps(judged, ensure_ascii=False) + "\n")
            print(f"Saved id={judged.get('id', idx)} | answer_correct={judged['judge_answer_correct']} | reasoning={judged['judge_reasoning_score']} | overall={judged['judge_overall_judgment']}")
    print(f"Done. Judged results saved to {OUTPUT_PATH}")


if __name__ == "__main__":
    main()

Writing /content/NLP/judge.py


In [18]:
%%writefile /content/NLP/metrics.py
import json
from pathlib import Path

INPUT_PATH = Path("outputs/judged_results_new.json")


def compute_metrics():
    total = 0
    correct = 0
    initial_correct = 0
    correction_count = 0
    regression_count = 0
    logic_density_sum = 0
    probe_score_sum = 0
    consistency_sum = 0
    with INPUT_PATH.open("r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            sample = json.loads(line)
            total += 1
            final_correct = sample.get("judge_answer_correct", False)
            if final_correct:
                correct += 1
            debate = sample.get("debate", {})
            if not isinstance(debate, dict):
                debate = {}
            initial = str(debate.get("initial_answer", ""))
            gold = str(debate.get("gold_answer", ""))
            initial_correct_flag = str(gold) in str(initial)
            if initial_correct_flag:
                initial_correct += 1
            if (not initial_correct_flag) and final_correct:
                correction_count += 1
            if initial_correct_flag and (not final_correct):
                regression_count += 1
            reasoning_score = sample.get("judge_reasoning_score", 0)
            turns = sample.get("num_turns", 1)
            logic_density = reasoning_score / max(turns, 1)
            logic_density_sum += logic_density
            probe_score = sample.get("judge_probe_response_score", 0) / 5
            probe_score_sum += probe_score
            consistency = sample.get("judge_consistency_score", 0) / 5
            consistency_sum += consistency
    if total == 0:
        print("No samples found.")
        return
    accuracy = correct / total
    initial_acc = initial_correct / total
    correction_rate = correction_count / total
    regression_rate = regression_count / total
    avg_logic_density = logic_density_sum / total
    avg_probe = probe_score_sum / total
    avg_consistency = consistency_sum / total
    print("\n=== METRICS ===")
    print(f"Samples: {total}")
    print(f"Final Accuracy: {accuracy:.2f}")
    print(f"Initial Accuracy: {initial_acc:.2f}")
    print(f"\nCorrection Rate: {correction_rate:.2f}")
    print(f"Regression Rate: {regression_rate:.2f}")
    print(f"\nAvg Logic Density: {avg_logic_density:.2f}")
    print(f"Avg Probe Effectiveness: {avg_probe:.2f}")
    print(f"Avg Consistency: {avg_consistency:.2f}")


if __name__ == "__main__":
    compute_metrics()

Writing /content/NLP/metrics.py


In [19]:

%%writefile /content/NLP/load_dataset.py
from datasets import load_dataset
import json
import re

dataset = load_dataset("gsm8k", "main")
test_data = dataset["test"]
print("Total test samples:", len(test_data))


def extract_answer(answer_text):
    match = re.search(r"####\s*(.*)", answer_text)
    return match.group(1).strip() if match else None


sample_ids = list(range(30))
samples = test_data.select(sample_ids)

with open("gsm8k_sample.jsonl", "w") as f:
    for i, s in enumerate(samples):
        data = {"id": i, "question": s["question"], "answer": extract_answer(s["answer"])}
        f.write(json.dumps(data, ensure_ascii=False) + "\n")

print(f"Saved {len(samples)} samples to gsm8k_sample.jsonl")

Writing /content/NLP/load_dataset.py


## 5. Load the GSM8K Dataset

In [20]:
%cd /content

%cd NLP
!python load_dataset.py

/content
/content/NLP
README.md: 7.93kB [00:00, 12.6MB/s]
main/train-00000-of-00001.parquet: 100% 2.31M/2.31M [00:00<00:00, 3.48MB/s]
main/test-00000-of-00001.parquet: 100% 419k/419k [00:00<00:00, 1.98MB/s]
Generating train split: 100% 7473/7473 [00:00<00:00, 165120.68 examples/s]
Generating test split: 100% 1319/1319 [00:00<00:00, 277363.23 examples/s]
Total test samples: 1319
Saved 30 samples to gsm8k_sample.jsonl


In [ ]:
!curl -s http://localhost:11434/api/tags | python3 -c "import sys,json; d=json.load(sys.stdin); print([m['name'] for m in d['models']])"

['qwen2.5:7b', 'llama3.1:8b']


## 6. Run the Debate (Proponent + Prober via Ollama)

This will take a while since 8B models run on the Colab GPU. Each sample does 1 initial call + 3 turns x 2 calls = 7 LLM calls.

In [21]:
!python run_debate.py

Running sample 0...
Running sample 1...
Running sample 2...
Running sample 3...
Running sample 4...
Running sample 5...
Running sample 6...
Running sample 7...
Running sample 8...
Running sample 9...
Running sample 10...
Running sample 11...
Running sample 12...
Running sample 13...
Running sample 14...
Running sample 15...
Running sample 16...
Running sample 17...
Running sample 18...
Running sample 19...
Running sample 20...
Running sample 21...
Running sample 22...
Running sample 23...
Running sample 24...
Running sample 25...
Running sample 26...
Running sample 27...
Running sample 28...
Running sample 29...
Saved results to outputs/debate_results_new.json


## 7. Run the Judge (Groq API - gpt-oss-20b)

In [22]:
!python judge.py

Saved id=0 | answer_correct=True | reasoning=5 | overall=pass
Saved id=1 | answer_correct=True | reasoning=2 | overall=fail
Saved id=2 | answer_correct=False | reasoning=1 | overall=fail
Saved id=3 | answer_correct=False | reasoning=2 | overall=fail
Saved id=4 | answer_correct=True | reasoning=5 | overall=pass
Saved id=5 | answer_correct=True | reasoning=5 | overall=pass
Saved id=6 | answer_correct=True | reasoning=5 | overall=pass
Saved id=7 | answer_correct=True | reasoning=2 | overall=fail
Saved id=8 | answer_correct=False | reasoning=1 | overall=fail
Saved id=9 | answer_correct=True | reasoning=5 | overall=pass
Saved id=10 | answer_correct=True | reasoning=5 | overall=pass
Saved id=11 | answer_correct=True | reasoning=5 | overall=pass
Saved id=12 | answer_correct=False | reasoning=2 | overall=fail
Saved id=13 | answer_correct=False | reasoning=1 | overall=fail
Saved id=14 | answer_correct=True | reasoning=2 | overall=fail
Saved id=15 | answer_correct=False | reasoning=1 | overall=f

## 8. Compute Metrics

In [23]:
!python metrics.py


=== METRICS ===
Samples: 30
Final Accuracy: 0.60
Initial Accuracy: 0.70

Correction Rate: 0.03
Regression Rate: 0.13

Avg Logic Density: 3.17
Avg Probe Effectiveness: 0.60
Avg Consistency: 0.65


## 9. Inspect Results

In [ ]:
import json

with open("outputs/judged_results_new.json") as f:
    for i, line in enumerate(f):
        if not line.strip():
            continue
        sample = json.loads(line)
        debate = sample.get("debate", {})
        print(
            f"ID={sample.get('id', i):>3} | "
            f"gold={debate.get('gold_answer', '?'):>6} | "
            f"initial={debate.get('initial_answer', '?'):>6} | "
            f"final={debate.get('final_answer', '?'):>6} | "
            f"judge={sample.get('judge_overall_judgment', '?')}"
        )